# Warmth annotation results

This notebook:
1. reads the LLM annotation results (`warmth_annotation_results.xlsx`)
2. keeps only the columns original-paragraph-in-german, translation, label and explanation
3. shows the % of paragraphs labelled true for WARMTH, together with their explanations
4. saves the paragraphs labelled false in a separate file

## 1. Packages

In [1]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", None)   # show full paragraphs and explanations

def show(table):
    """Show a table with left-aligned text that wraps inside each cell."""
    return table.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap", "min-width": "250px"}) \
        .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])


## 2. Settings

In [7]:
input_path       = Path("warmth_annotation_results.xlsx")
false_output     = Path("warmth_false.xlsx")

keep_columns = ["row-id", "original-paragraph-in-german", "translation", "label", "explanation"]

## 3. Load the data and keep the needed columns

Column names are matched without caring about upper/lower case, spaces or underscores, so `Original Paragraph in German` or `original_paragraph_in_german` also work. All other columns (for example `json_ok`) are dropped.

In [8]:
df = pd.read_excel(input_path)
print("Columns in the file:", list(df.columns))


def normalise(name):
    return str(name).strip().lower().replace("_", "-").replace(" ", "-")


column_map = {col: normalise(col) for col in df.columns}
df = df.rename(columns=column_map)
df = df.rename(columns={"text-block": "original-paragraph-in-german"})   # name used in the results file

missing = [col for col in keep_columns if col not in df.columns]
if missing:
    raise KeyError(f"Missing columns: {missing}. Columns found: {list(df.columns)}")

df = df[keep_columns]
print(len(df), "rows")
show(df.head())

Columns in the file: ['row_id', 'article_id', 'pub', 'par_index', 'group', 'length', 'text_block', 'translation', 'model', 'seed', 'retried', 'needs_check', 'json_ok', 'placeholder_ok', 'length_ratio', 'length_ok', 'tokens_prompt', 'tokens_response', 'seconds', 'id', 'raw_output', 'label', 'explanation']
5000 rows


AttributeError: The '.style' accessor requires jinja2

## 4. Clean the label column

The LLM output can be `True`, `TRUE`, `true`, `1`, `yes` and so on. Everything is turned into a real True/False value. Labels that cannot be read are shown below so they can be fixed by hand.

In [ ]:
true_values  = {"true", "1", "yes", "ja", "wahr"}
false_values = {"false", "0", "no", "nein", "falsch"}


def to_bool(value):
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in true_values:
        return True
    if text in false_values:
        return False
    return pd.NA


df["label"] = df["label"].map(to_bool).astype("boolean")

unreadable = df[df["label"].isna()]
print(len(unreadable), "rows with a missing or unreadable label")
show(unreadable)

4 rows with a missing or unreadable label


,original-paragraph-in-german,translation,label,explanation
2202,"Außenseiter [Andere Gruppe] macht eigentlich ein gutes Spiel, hält kämpferisch dagegen, kann die Poweroffensive der [Gruppe 1] aber einfach nicht stoppen. Immer, wenn Mbappé über links oder Dembele über rechts Tempo aufnehmen, wird es gefährlich. Dabei übertreiben es die beiden auch gerne mal, spielen mit der Hacke oder machen sieben Übersteiger hintereinander - auch das kennt man. Trotzdem ist es eine Show, wenn sie in Wallung kommen. Lewandowski, der kurz vor Schluss per Elfmeter noch den Ehrentreffer erzielt, muss nach Hause fahren - weil Mbappé mit seinen WM-Treffern acht und neun zuvor den Deckel draufmacht (74., 90.+1). Beim ersten großen Turnier 2018 war der [Gruppe 1] gleich Weltmeister geworden. Diesmal feiert er zwei Tage nach dem WM-Finale seinen 24. Geburtstag - mit dem goldenen Pokal? 3:1","Underdog [Andere Gruppe] actually plays a good game, fighting back tenaciously, but simply cannot stop the powerful offense of [Gruppe 1]. Whenever Mbappé picks up speed on the left or Dembele on the right, it becomes dangerous. At the same time, the two of them also like to overdo it, playing with their heels or performing seven stepovers in a row - that is also well known. Nevertheless, it is a show when they get into their rhythm. Lewandowski, who scores the consolation goal via penalty shortly before the end, has to go home - because Mbappé, with his World Cup goals eight and nine previously, puts the lid on it (74th, 90.+1). At his first major tournament in 2018, [Gruppe 1] immediately became world champions. This time, he celebrates his 24th birthday two days after the World Cup final - with the golden trophy? 3:1",,nan
2203,"Ohne Medaille: Die deutsche Siegesserie im olympischen Eiskanal von Yanqing ist gerissen. Für die Winterbergerin Laura Nolte reichte es im Monobob nur zu Platz vier. Europameisterin Mariama Jamanka aus Oberhof fuhr in dem neuen Wettbewerb auf Rang 13. Zuvor hatten sowohl die deutschen Rodler als auch die Skeleton-Piloten alle ihre Wettbewerbe und neben sechs Goldmedaillen noch dreimal Silber gewonnen. Erste Monobob-Olympiasiegerin ist Weltmeisterin Kaillie Humphries aus den USA, die am Ende 1,54 Sekunden vor ihrer Teamkollegin Elana Meyers Taylor. Bronze sicherte sich die Kanadierin Christine de Bruin, die 0,30 Sekunden schneller als Nolte war. Gold-Tanz: Die Eistänzer Gabriella Papadakis/Guillaume Cizeron aus Frankreich sind mit dem Olympiasieg von Peking am Ziel ihrer Träume angekommen. Mit 226,98 Punkten verwiesen die viermaligen Weltmeister in der Kür die russischen Europameister Wiktoria Sinizina/Nikita Kazalapow (220,51) auf den Silberrang. Dritte wurden die [Gruppe 1] Madison Hubbell/Zachary Donohue mit 218,02 Punkten. Pech hatten Katharina Müller und Tim Dieck (Berlin/Dortmund) bei ihrem Olympia-Debüt. Die zweimaligen deutschen Meister mussten bei den Rhythmischen Tänzen am Samstag mit der undankbaren Startnummer eins auf das Eis und verfehlten mit dem 21. Platz den Einzug in die Kür der 20 besten Paare. Null-Covid: In der geschlossenen Olympia-Blase sind erstmals keine neuen Corona-Infektionen aufgetreten. Wie die Organisatoren mitteilten, fielen sämtliche 69.872 am Vortag durchgeführte Tests negativ aus. Drei weitere Infektionen wurden bei anreisenden Beteiligten am Pekinger Flughafen entdeckt. Der Trend neuer Infektionen zeigt seit Tagen nach unten. Während nach Beginn der Erfassung am 23. Januar zeitweise über 30 Infektionen pro Tag registriert wurden, lag die Zahl neuer Fälle zuletzt stets im einstelligen Bereich. Seit Beginn der Anreise zu den Winterspielen wurden insgesamt 432 Infektionen nachgewiesen.","No medal: Germany's winning streak in Yanqing's Olympic ice track has been broken. For Winterberg's Laura Nolte, fourth place was all that was achieved in the monobob. European champion Mariama Jamanka from Oberhof finished in 13th place in the new competition. Previously, both the German lugers and the skeleton pilots h

## 5. Percentage of WARMTH = true

In [ ]:
n_total = df["label"].notna().sum()
n_true  = (df["label"] == True).sum()
n_false = (df["label"] == False).sum()

print(f"WARMTH true:  {n_true} of {n_total} ({n_true / n_total:.1%})")
print(f"WARMTH false: {n_false} of {n_total} ({n_false / n_total:.1%})")

WARMTH true:  226 of 4996 (4.5%)
WARMTH false: 4770 of 4996 (95.5%)


### Paragraphs labelled true, with the explanation

In [ ]:
warmth_true = df[df["label"] == True].reset_index(drop=True)
show(warmth_true[["original-paragraph-in-german", "translation", "explanation"]])

,original-paragraph-in-german,translation,explanation
0,"Das Minkalab in den Bergen Kolumbiens ist eine Kombination aus Bio-Bauernhof, Naturschutzgebiet und Begegnungsstätte und versteht sich als Plattform für die Stärkung von Minderheiten und die Förderung von interkulturellem Austausch im Sinne aller Lebewesen. „Minka“ ist Quechua, eine Sprache, die im Andenraum Südamerikas gesprochen wird, und bedeutet Zusammenarbeit. Denn nur die Kooperation kann uns in unserer Entwicklung weiterbringen. Isoliert sind wir machtlos. Jeden Monat werden Workshops veranstaltet, Bauprojekte, Werkstätten, Diskussionen zu den Themen Umwelt, Kunst, Technologie, Tradition und ganzheitliche Medizin. Jung und Alt kommen zusammen, Landwirte und Künstler, [Andere Gruppe] und [Gruppe 1], um die Probleme des jeweils anderen zu verstehen und gemeinsam nach Lösungen zu suchen. Bei den Treffen geht es neben dem Austausch über den Verlust der biologischen Vielfalt, die Folgen der globalisierten Wirtschaft und die Auswirkungen der Monokultur und des Mono-Denkens um konkrete Initiativen wie biologisches Bauen, Selbstversorgung, natürliche Architektur, indigene Rituale, die Einrichtung einer Saatgutbank oder die Entwicklung von Programmen zur Förderung der Verwendung [Andere Gruppe] Heilpflanzen zur Vorbeugung und Heilung von Krankheiten. Alle Projekte sind darauf ausgerichtet, mit bestehenden sozialen Einrichtungen zusammenzuarbeiten und die Entwicklung eines aktiven und autonomen Netzwerks zu fördern, in dessen Zentrum der Erhalt der Gesundheit, der Schutz der Umwelt und die Entwicklung einer friedlichen Gemeinschaft steht.","The Minkalab in the mountains of Colombia is a combination of an organic farm, a nature reserve, and a meeting place, and views itself as a platform for strengthening minorities and promoting intercultural exchange for the sake of all living beings. ""Minka"" is Quechua, a language spoken in the Andean region of South America, and means cooperation. Because only cooperation can move us forward in our development. Isolated, we are powerless. Every month, workshops are held, as well as construction projects, workshops, and discussions on the topics of environment, art, technology, tradition, and holistic medicine. Young and old come together, farmers and artists, [Andere Gruppe] and [Gruppe 1], to understand each other's problems and to seek solutions together. In addition to exchanging views on the loss of biodiversity, the consequences of the globalized economy, and the effects of monoculture and mono-thinking, the meetings focus on concrete initiatives such as organic construction, self-sufficiency, natural architecture, indigenous rituals, the establishment of a seed bank, or the development of programs to promote the use of [Andere Gruppe] medicinal plants for the prevention and healing of diseases. All projects are aimed at working with existing social institutions and promoting the development of an active and autonomous network, at the center of which lies the preservation of health, the protection of the environment, and the development of a peaceful community.","The text portrays [Gruppe 1] as participating in a community focused on cooperation, seeking solutions together, and developing a peaceful community, which reflects positive intentions and sociability."
1,"Wellbrock hatte seinen besonderen Fokus auf die 1500 Meter gelegt. Nachdem er den Vorlauf gegen die starke Konkurrenz gewonnen hatte, legte er Wert auf schnelle Regeneration. Das war im Vorfeld der Wettbewerbe von [Andere Gruppe] ein von Bundestrainer Bernd Berkhahn entwickeltes neues Trainingsziel gewesen. Auch die Lockerheit sollte nicht vernachlässigt werden. [Gruppe 1] dreht auf So sah man Wellbrock auch am Samstagmorgen vor der Vormittags-Session ein paar schnelle Bahnen ziehen, mit Berkhahn immer wieder diskutieren, aber auch als Spaßkanone. Als Freistil-Schwimmer Rafael Miroslaw beispielsweise eine Spurt-Einheit über 25 Meter schwamm, hampelte Wellbrock mit den A

## 6. Save the false paragraphs in a separate file

In [ ]:
warmth_false = df[df["label"] == False].reset_index(drop=True)

false_output.parent.mkdir(parents=True, exist_ok=True)
warmth_false.to_excel(false_output, index=False)
print(f"Saved {len(warmth_false)} rows to {false_output}")

Saved 4770 rows to data/warmth_false.xlsx
